# Chapter 18
## Bistability Resulting from Rebound Firing
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter18.ipynb)

## About this chapter

A resting state and a repetitive-firing state can coexist at the same
applied current -- a small perturbation decides which one a trajectory
ends up in. This chapter finds the (typically unstable) fixed point
separating the two basins, by bisecting the current-balance equation, then
integrates from just on either side of it. Two slow currents are studied:
$I_h$ (hyperpolarization-activated, deinactivates on hyperpolarization,
produces rebound firing) and $I_M$ (M-current, produces spike-frequency
adaptation that can also support bistability). "Gates" variants plot the
relevant slow variable against its instantaneous $x_\infty(v)$ curve;
"limited" variants artificially clamp the slow variable's drift, isolating
which direction of its change is responsible for the trajectory choosing
its basin.

See [`README.md`](chapter18.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact

## The $I_h$ Current: Steady State and Time Constant

In [ ]:
def r_inf(v):
    return 1.0 / (1 + np.exp((v + 84) / 10.2))


def tau_r(v):
    return 1.0 / (np.exp(-14.59 - 0.086 * v) + np.exp(-1.87 + 0.0701 * v))


def simulate_h_current(v=None):
    if v is None:
        v = np.arange(-100, 51)
    return v, r_inf(v), tau_r(v)


def plot_h_current(v, r, t):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(v, r, '-k', linewidth=2)
    ax[0].set_xlabel('$v$ [mV]')
    ax[0].set_ylabel(r'$r_\infty(v)$')
    ax[0].set_xlim(-100, 50)
    ax[0].set_ylim(0, 1)

    ax[1].plot(v, t, '-k', linewidth=2)
    ax[1].set_xlabel('$v$ [mV]')
    ax[1].set_ylabel(r'$\tau_r$ [ms]')
    ax[1].set_xlim(-100, 50)
    ax[1].set_ylim(0, 1000)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_h_current(*simulate_h_current())

## Modified $\tau_r$

A book-specific modification that flattens $\tau_r$ below $v=-80$ mV
(the unmodified curve would make the resting timescale unrealistically
long there).

In [ ]:
def tau_r_modified(v):
    tau = tau_r(v)
    factor = (0.05 * v + 5) * (v < -80) + (v >= -80)
    return factor ** 2 * tau


def simulate_modified_tau_r(v=None):
    if v is None:
        v = np.arange(-100, 51)
    return v, tau_r(v), tau_r_modified(v)


def plot_modified_tau_r(v, tau_original, tau_modified):
    plt.figure(figsize=(7, 5))
    plt.plot(v, tau_modified, '--b', linewidth=2)
    plt.plot(v, tau_original, '-k', linewidth=2)
    plt.xlabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_modified_tau_r(*simulate_modified_tau_r())

## HH Gating (shared by the `HH_BISTABLE*` examples below)

In [ ]:
def hh_bistable_alpha_m(v):
    with np.errstate(divide='ignore', invalid='ignore'):
        out = (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))
    return np.where(np.abs(v + 45) > 1e-8, out, 1.0)


def hh_bistable_alpha_h(v):
    return 0.07 * exp(-(v + 70) / 20)


def hh_bistable_alpha_n(v):
    return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)


def hh_bistable_beta_h(v):
    return 1.0 / (exp(-(v + 40) / 10) + 1)


def hh_bistable_beta_m(v):
    return 4 * exp(-(v + 70) / 18)


def hh_bistable_beta_n(v):
    return 0.125 * exp(-(v + 70) / 80)


def hh_bistable_m_inf(v):
    return hh_bistable_alpha_m(v) / (hh_bistable_alpha_m(v) + hh_bistable_beta_m(v))


def hh_bistable_h_inf(v):
    return hh_bistable_alpha_h(v) / (hh_bistable_alpha_h(v) + hh_bistable_beta_h(v))


def hh_bistable_n_inf(v):
    return hh_bistable_alpha_n(v) / (hh_bistable_alpha_n(v) + hh_bistable_beta_n(v))


def hh_bistable_resting_point(g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0, i_ext=8.0):
    """dv/dt at the given i_ext, for the resting-state bisection"""
    def f(v):
        return (g_na * hh_bistable_m_inf(v) ** 3 * hh_bistable_h_inf(v) * (v_na - v)
                + g_k * hh_bistable_n_inf(v) ** 4 * (v_k - v) + g_l * (v_l - v) + i_ext)

    v_left, v_right = -100.0, 50.0
    while v_right - v_left > 1e-12:
        v_c = (v_left + v_right) / 2
        if f(v_c) * f(v_right) < 0:
            v_left = v_c
        else:
            v_right = v_c
    v_star = (v_left + v_right) / 2
    return v_star, hh_bistable_m_inf(v_star), hh_bistable_h_inf(v_star), hh_bistable_n_inf(v_star)

## HH Bistability: Resting vs. Firing

In [ ]:
def simulate_hh_bistable(c=1.0, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0,
                          i_ext=8.0, t_final=40.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star = hh_bistable_resting_point(g_na, g_k, g_l, v_na, v_k, v_l, i_ext)

    def simulate(v0, m0, h0, n0):
        """Heun/RK2 integration (plain floats) of the full (v,m,h,n) HH model"""
        v, m, h, n = v0, m0, h0, n0
        v_trace = np.empty(m_steps + 1)
        v_trace[0] = v
        for k in range(m_steps):
            v_inc = (g_na * m ** 3 * h * (v_na - v) + g_k * n ** 4 * (v_k - v) + g_l * (v_l - v) + i_ext) / c
            m_inc = hh_bistable_alpha_m(v) * (1 - m) - hh_bistable_beta_m(v) * m
            h_inc = hh_bistable_alpha_h(v) * (1 - h) - hh_bistable_beta_h(v) * h
            n_inc = hh_bistable_alpha_n(v) * (1 - n) - hh_bistable_beta_n(v) * n

            v_tmp = v + dt05 * v_inc
            m_tmp = m + dt05 * m_inc
            h_tmp = h + dt05 * h_inc
            n_tmp = n + dt05 * n_inc

            v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                     + g_l * (v_l - v_tmp) + i_ext) / c
            m_inc = hh_bistable_alpha_m(v_tmp) * (1 - m_tmp) - hh_bistable_beta_m(v_tmp) * m_tmp
            h_inc = hh_bistable_alpha_h(v_tmp) * (1 - h_tmp) - hh_bistable_beta_h(v_tmp) * h_tmp
            n_inc = hh_bistable_alpha_n(v_tmp) * (1 - n_tmp) - hh_bistable_beta_n(v_tmp) * n_tmp

            v = v + dt * v_inc
            m = m + dt * m_inc
            h = h + dt * h_inc
            n = n + dt * n_inc
            v_trace[k + 1] = v
        return v_trace

    v_rest = simulate(v_star, m_star, h_star, n_star)
    v_fire = simulate(v_star + 5, m_star, h_star, n_star)
    return v_rest, v_fire, v_star, t_final, dt


def plot_bistable_rest_fire(v_rest, v_fire, t_final, dt, vlim=(-100, 50)):
    t = np.arange(len(v_rest)) * dt
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, v_rest, '-k', linewidth=2)
    ax[0].set_ylabel('$v$ [mV]')
    ax[0].set_xlim(0, t_final)
    ax[0].set_ylim(*vlim)

    ax[1].plot(t, v_fire, '-k', linewidth=2)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel('$v$ [mV]')
    ax[1].set_xlim(0, t_final)
    ax[1].set_ylim(*vlim)

    plt.tight_layout()
    plt.show()

In [ ]:
v_rest, v_fire, v_star, t_final, dt = simulate_hh_bistable()
plot_bistable_rest_fire(v_rest, v_fire, t_final, dt)

## HH Bistability: Gating Variables

In [ ]:
def simulate_hh_bistable_gates(c=1.0, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0,
                                i_ext=8.0, t_final=40.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star = hh_bistable_resting_point(g_na, g_k, g_l, v_na, v_k, v_l, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0] = v_star + 5, m_star, h_star, n_star

    for k in range(m_steps):
        v_inc = (g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_k * n[k] ** 4 * (v_k - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        m_inc = hh_bistable_alpha_m(v[k]) * (1 - m[k]) - hh_bistable_beta_m(v[k]) * m[k]
        h_inc = hh_bistable_alpha_h(v[k]) * (1 - h[k]) - hh_bistable_beta_h(v[k]) * h[k]
        n_inc = hh_bistable_alpha_n(v[k]) * (1 - n[k]) - hh_bistable_beta_n(v[k]) * n[k]

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m[k] + dt05 * m_inc
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc

        v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        m_inc = hh_bistable_alpha_m(v_tmp) * (1 - m_tmp) - hh_bistable_beta_m(v_tmp) * m_tmp
        h_inc = hh_bistable_alpha_h(v_tmp) * (1 - h_tmp) - hh_bistable_beta_h(v_tmp) * h_tmp
        n_inc = hh_bistable_alpha_n(v_tmp) * (1 - n_tmp) - hh_bistable_beta_n(v_tmp) * n_tmp

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m[k] + dt * m_inc
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc

    return v, m, h, n, m_star, h_star, n_star, t_final


def plot_hh_bistable_gates(v, m, h, n, m_star, h_star, n_star, t_final):
    t = np.arange(len(v)) * (t_final / (len(v) - 1))
    fig, ax = plt.subplots(3, figsize=(7, 8), sharex=True)

    ax[0].plot(t, m, '-k', linewidth=2)
    ax[0].plot(t, hh_bistable_m_inf(v), '--b', linewidth=2)
    ax[0].plot([0, t_final], [m_star, m_star], '-r', linewidth=2)
    ax[0].set_ylabel('$m$')
    ax[0].set_ylim(0, 1)

    ax[1].plot(t, h, '-k', linewidth=2)
    ax[1].plot(t, hh_bistable_h_inf(v), '--b', linewidth=2)
    ax[1].plot([0, t_final], [h_star, h_star], '-r', linewidth=2)
    ind = np.where(h > h_star)[0]
    ax[1].plot(t[ind], np.zeros(len(ind)), '.m', markersize=8)
    ax[1].set_ylabel('$h$')
    ax[1].set_ylim(0, 1)

    ax[2].plot(t, n, '-k', linewidth=2)
    ax[2].plot(t, hh_bistable_n_inf(v), '--b', linewidth=2)
    ax[2].plot([0, t_final], [n_star, n_star], '-r', linewidth=2)
    ind = np.where(n < n_star)[0]
    ax[2].plot(t[ind], np.zeros(len(ind)), '.m', markersize=8)
    ax[2].set_ylabel('$n$')
    ax[2].set_xlabel('$t$ [ms]')
    ax[2].set_ylim(0, 1)
    ax[2].set_xlim(0, t_final)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_bistable_gates(*simulate_hh_bistable_gates())

## HH Bistability: Limited $n$

Clamps $n$ so it can never fall below $n_\ast$, isolating that this
decline is what lets the trajectory return to rest.

In [ ]:
def simulate_hh_bistable_limited_n(c=1.0, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0,
                                    i_ext=8.0, t_final=40.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star = hh_bistable_resting_point(g_na, g_k, g_l, v_na, v_k, v_l, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0] = v_star + 5, m_star, h_star, n_star

    for k in range(m_steps):
        v_inc = (g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_k * n[k] ** 4 * (v_k - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        m_inc = hh_bistable_alpha_m(v[k]) * (1 - m[k]) - hh_bistable_beta_m(v[k]) * m[k]
        h_inc = hh_bistable_alpha_h(v[k]) * (1 - h[k]) - hh_bistable_beta_h(v[k]) * h[k]
        n_inc = hh_bistable_alpha_n(v[k]) * (1 - n[k]) - hh_bistable_beta_n(v[k]) * n[k]

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m[k] + dt05 * m_inc
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc

        v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        m_inc = hh_bistable_alpha_m(v_tmp) * (1 - m_tmp) - hh_bistable_beta_m(v_tmp) * m_tmp
        h_inc = hh_bistable_alpha_h(v_tmp) * (1 - h_tmp) - hh_bistable_beta_h(v_tmp) * h_tmp
        n_inc = hh_bistable_alpha_n(v_tmp) * (1 - n_tmp) - hh_bistable_beta_n(v_tmp) * n_tmp

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m[k] + dt * m_inc
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = max(n_star, n[k] + dt * n_inc)

    return v, t_final, dt


def plot_voltage_trace_bistable(v, t_final, dt, vlim=(-100, 50)):
    t = np.arange(len(v)) * dt
    plt.figure(figsize=(7, 3.5))
    plt.plot(t, v, '-k', linewidth=2)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.xlim(0, t_final)
    plt.ylim(*vlim)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_voltage_trace_bistable(*simulate_hh_bistable_limited_n())

## RTM Gating (shared by the `RTM_WITH_I_H*`/`RTM_WITH_I_M*`/`RTM_VOLTAGE_TRACE_WITH_I_H` examples below)

Byte-identical to `mnd.core`'s RTM family, imported directly.

In [ ]:
from mnd.core import alpha_h, alpha_m, alpha_n, beta_h, beta_m, beta_n, h_inf, m_inf, n_inf


def rtm_i_h_resting_point(g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                           g_h=1.0, v_h=-32.9, i_ext=-3.19):
    def f(v):
        return (g_na * m_inf(v) ** 3 * h_inf(v) * (v_na - v) + g_k * n_inf(v) ** 4 * (v_k - v)
                + g_l * (v_l - v) + g_h * r_inf(v) * (v_h - v) + i_ext)

    v_grid = -100 + np.arange(100001) / 100000 * 150
    with np.errstate(invalid='ignore'):
        f_grid = f(v_grid)
    ind = np.where(f_grid[:-1] * f_grid[1:] <= 0)[0]
    v_left = v_grid[ind].min()
    v_right = v_grid[ind + 1].min()
    while v_right - v_left > 1e-12:
        v_c = (v_left + v_right) / 2
        if f(v_c) * f(v_left) <= 0:
            v_right = v_c
        else:
            v_left = v_c
    v_star = (v_left + v_right) / 2
    return v_star, m_inf(v_star), h_inf(v_star), n_inf(v_star), r_inf(v_star)


def rtm_i_m_resting_point(g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                           g_m=0.2, i_ext=0.506):
    def f(v):
        return (g_na * m_inf(v) ** 3 * h_inf(v) * (v_na - v) + g_k * n_inf(v) ** 4 * (v_k - v)
                + g_l * (v_l - v) + g_m * w_inf(v) * (v_k - v) + i_ext)

    v_grid = -100 + np.arange(100001) / 100000 * 150
    with np.errstate(invalid='ignore'):
        f_grid = f(v_grid)
    ind = np.where(f_grid[:-1] * f_grid[1:] <= 0)[0].min()
    v_left, v_right = v_grid[ind], v_grid[ind + 1]
    while v_right - v_left > 1e-14:
        v_c = (v_left + v_right) / 2
        if f(v_c) * f(v_right) <= 0:
            v_left = v_c
        else:
            v_right = v_c
    v_star = (v_left + v_right) / 2
    return v_star, m_inf(v_star), h_inf(v_star), n_inf(v_star), w_inf(v_star)

## RTM Voltage Trace with $I_h$

A step of hyperpolarizing current followed by release triggers a rebound
spike via $I_h$ deinactivation.

In [ ]:
def simulate_rtm_voltage_trace_with_i_h(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                         v_k=-100.0, v_na=50.0, v_l=-67.0,
                                         g_h=1.0, v_h=-32.9, t_final=300.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    r = np.zeros(m_steps + 1)
    v[0], m[0], h[0], n[0], r[0] = -70.0, 0.0, 1.0, 0.0, 0.5

    for k in range(m_steps):
        i_ext = -4.0 if (t_final / 3 <= k * dt < 2 * t_final / 3) else 0.0

        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k])
                 + g_l * (v_l - v[k]) + g_h * r[k] * (v_h - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]
        r_inc = (r_inf(v[k]) - r[k]) / tau_r(v[k])

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        r_tmp = r[k] + dt05 * r_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_h * r_tmp * (v_h - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        r_inc = (r_inf(v_tmp) - r_tmp) / tau_r(v_tmp)

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        r[k + 1] = r[k] + dt * r_inc

    return v, t_final, dt

In [ ]:
plot_voltage_trace_bistable(*simulate_rtm_voltage_trace_with_i_h(), vlim=(-100, 50))

## RTM Bistability with $I_h$: Resting vs. Firing

In [ ]:
def simulate_rtm_with_i_h_bistable(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                    v_k=-100.0, v_na=50.0, v_l=-67.0,
                                    g_h=1.0, v_h=-32.9, i_ext=-3.19, t_final=3000.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star, r_star = rtm_i_h_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, g_h, v_h, i_ext)

    def simulate(v0, m0, h0, n0, r0):
        """Heun/RK2 integration (plain floats) of the RTM model with an added
        I_h current, m quasi-static"""
        v, m, h, n, r = v0, m0, h0, n0, r0
        v_trace = np.empty(m_steps + 1)
        v_trace[0] = v
        for k in range(m_steps):
            v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                     + g_h * r * (v_h - v) + i_ext) / c
            n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n
            h_inc = alpha_h(v) * (1 - h) - beta_h(v) * h
            r_inc = (r_inf(v) - r) / tau_r(v)

            v_tmp = v + dt05 * v_inc
            m_tmp = m_inf(v_tmp)
            h_tmp = h + dt05 * h_inc
            n_tmp = n + dt05 * n_inc
            r_tmp = r + dt05 * r_inc

            v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                     + g_l * (v_l - v_tmp) + g_h * r_tmp * (v_h - v_tmp) + i_ext) / c
            h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
            n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
            r_inc = (r_inf(v_tmp) - r_tmp) / tau_r(v_tmp)

            v = v + dt * v_inc
            m = m_inf(v)
            h = h + dt * h_inc
            n = n + dt * n_inc
            r = r + dt * r_inc
            v_trace[k + 1] = v
        return v_trace

    v_rest = simulate(v_star + 0.01, m_star, h_star, n_star, r_star)
    v_fire = simulate(v_star + 1, m_star, h_star, n_star, r_star)
    return v_rest, v_fire, v_star, t_final, dt

In [ ]:
v_rest, v_fire, v_star, t_final, dt = simulate_rtm_with_i_h_bistable()
plot_bistable_rest_fire(v_rest, v_fire, t_final, dt, vlim=(-100, 50))

## RTM Bistability with $I_h$: Gating Variables

In [ ]:
def simulate_rtm_with_i_h_bistable_gates(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                          v_k=-100.0, v_na=50.0, v_l=-67.0,
                                          g_h=1.0, v_h=-32.9, i_ext=-3.19, t_final=2000.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star, r_star = rtm_i_h_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, g_h, v_h, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    r = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0], r[0] = v_star + 1, m_star, h_star, n_star, r_star

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_l * (v_l - v[k])
                 + g_h * r[k] * (v_h - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]
        r_inc = (r_inf(v[k]) - r[k]) / tau_r(v[k])

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        r_tmp = r[k] + dt05 * r_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_h * r_tmp * (v_h - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        r_inc = (r_inf(v_tmp) - r_tmp) / tau_r(v_tmp)

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        r[k + 1] = r[k] + dt * r_inc

    return v, h, n, r, h_star, n_star, r_star, t_final


def plot_rtm_with_i_h_bistable_gates(v, h, n, r, h_star, n_star, r_star, t_final):
    t = np.arange(len(v)) * (t_final / (len(v) - 1))
    fig, ax = plt.subplots(3, figsize=(7, 8), sharex=True)

    ax[0].plot(t, h, '-k', linewidth=2)
    ax[0].plot([0, t_final], [h_star, h_star], '-r', linewidth=2)
    ax[0].plot(t, h_inf(v), '--b', linewidth=2)
    ax[0].set_ylabel('$h$')
    ax[0].set_ylim(0.98, 1)

    ax[1].plot(t, n, '-k', linewidth=2)
    ax[1].plot([0, t_final], [n_star, n_star], '-r', linewidth=2)
    ax[1].plot(t, n_inf(v), '--b', linewidth=2)
    ax[1].set_ylabel('$n$')
    ax[1].set_ylim(0, 0.1)

    ax[2].plot(t, r, '-k', linewidth=2)
    ax[2].plot([0, t_final], [r_star, r_star], '-r', linewidth=2)
    ax[2].plot(t, r_inf(v), '--b', linewidth=2)
    ind = np.where(r > r_star)[0]
    ax[2].plot(t[ind], np.zeros(len(ind)), '.m', markersize=8)
    ax[2].set_ylabel('$r$')
    ax[2].set_xlabel('$t$ [ms]')
    ax[2].set_ylim(0, 0.2)
    ax[2].set_xlim(0, t_final)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_with_i_h_bistable_gates(*simulate_rtm_with_i_h_bistable_gates())

## RTM Bistability with $I_h$: Limited $r$

Clamps $r$ so it can never rise above $r_\ast$, isolating that this
recovery is what lets the trajectory settle back onto the resting state.

In [ ]:
def simulate_rtm_with_i_h_limited_r(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                     v_k=-100.0, v_na=50.0, v_l=-67.0,
                                     g_h=1.0, v_h=-32.9, i_ext=-3.19, t_final=3000.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star, r_star = rtm_i_h_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, g_h, v_h, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    r = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0], r[0] = v_star + 1, m_star, h_star, n_star, r_star

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_l * (v_l - v[k])
                 + g_h * r[k] * (v_h - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]
        r_inc = (r_inf(v[k]) - r[k]) / tau_r(v[k])

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        r_tmp = r[k] + dt05 * r_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_h * r_tmp * (v_h - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        r_inc = (r_inf(v_tmp) - r_tmp) / tau_r(v_tmp)

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        r[k + 1] = min(r[k] + dt * r_inc, r_star)

    return v, t_final, dt

In [ ]:
plot_voltage_trace_bistable(*simulate_rtm_with_i_h_limited_r(), vlim=(-100, 50))

## RTM $I_h$ F-I Curve (Hysteresis)

In [ ]:
def simulate_rtm_f_i_curve_with_i_h(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                     v_k=-100.0, v_na=50.0, v_l=-67.0,
                                     g_h=1.0, v_h=-32.9, dt=0.01, t_max=2000000.0, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def run_to_frequency(i_ext, v, m, h, n, r):
        """f=0 if v settles to rest, else the frequency from the 3rd/4th
        spike interval."""
        win_maxv = win_minv = v
        win_maxm = win_minm = m
        win_maxh = win_minh = h
        win_maxn = win_minn = n
        win_maxr = win_minr = r
        num_spikes = 0
        t_spikes = []

        for k in range(1, t_max_steps + 1):
            v_prev = v
            v_inc = (g_na * m ** 3 * h * (v_na - v) + g_k * n ** 4 * (v_k - v) + g_l * (v_l - v)
                     + g_h * r * (v_h - v) + i_ext) / c
            h_inc = alpha_h(v) * (1 - h) - beta_h(v) * h
            n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n
            r_inc = (r_inf(v) - r) / tau_r(v)

            v_tmp = v + dt05 * v_inc
            m_tmp = m_inf(v_tmp)
            h_tmp = h + dt05 * h_inc
            n_tmp = n + dt05 * n_inc
            r_tmp = r + dt05 * r_inc

            v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                     + g_h * r_tmp * (v_h - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
            h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
            n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
            r_inc = (r_inf(v_tmp) - r_tmp) / tau_r(v_tmp)

            v = v + dt * v_inc
            m = m_inf(v)
            h = h + dt * h_inc
            n = n + dt * n_inc
            r = r + dt * r_inc

            win_maxv, win_minv = max(win_maxv, v), min(win_minv, v)
            win_maxm, win_minm = max(win_maxm, m), min(win_minm, m)
            win_maxh, win_minh = max(win_maxh, h), min(win_minh, h)
            win_maxn, win_minn = max(win_maxn, n), min(win_minn, n)
            win_maxr, win_minr = max(win_maxr, r), min(win_minr, r)

            if v < -20 and v_prev >= -20:
                num_spikes += 1
                t_spikes.append((k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v))
                if num_spikes == 4:
                    return 1000 / (t_spikes[3] - t_spikes[2]), v, m, h, n, r

            if k % N == 0:
                if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
                   (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
                   (win_maxh - win_minh) < 1e-4 * abs(win_maxh + win_minh) and \
                   (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn) and \
                   (win_maxr - win_minr) < 1e-4 * abs(win_maxr + win_minr):
                    return 0.0, v, m, h, n, r
                win_maxv = win_minv = v
                win_maxm = win_minm = m
                win_maxh = win_minh = h
                win_maxn = win_minn = n
                win_maxr = win_minr = r

        raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")

    if i_ext_vec is None:
        i_ext_low, i_ext_high = -3.2, -3.185
        i_ext_vec = i_ext_low + np.arange(31) / 30 * (i_ext_high - i_ext_low)

    f_forward = np.zeros(len(i_ext_vec))
    v, m, h, n, r = -70.0, m_inf(-70.0), 0.7, 0.6, 0.2
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, m, h, n, r = run_to_frequency(i_ext, v, m, h, n, r)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, m, h, n, r = run_to_frequency(i_ext_vec[ijk], v, m, h, n, r)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec

In [ ]:
# Very slow (several minutes to tens of minutes): several i_ext points sit
# close to the two bistability thresholds, where settling/spiking takes a
# very long simulated time.
f_forward, f_backward, I_c, I_star, i_ext_vec = simulate_rtm_f_i_curve_with_i_h()
print(f"I_c = {I_c}")
print(f"I_star = {I_star}")
plt.figure(figsize=(7, 3.5))
plt.plot(i_ext_vec, f_forward, '.k', markersize=15, label='forward')
plt.plot(i_ext_vec, f_backward, 'ok', markersize=10, markerfacecolor='none', linewidth=1, label='backward')
plt.xlim(i_ext_vec.min(), i_ext_vec.max())
plt.ylim(0, max(f_forward.max(), f_backward.max()) * 1.1)
plt.xlabel('$I$')
plt.ylabel('$f$')
plt.tight_layout()
plt.show()

## $I_M$ Steady State (imported)

`w_inf`/`tau_w` for the M-current match `python/chapter09.ipynb`'s
`w_inf`/`tau_w` exactly; redefined here so this notebook is self-contained.

In [ ]:
def w_inf(v):
    return 1.0 / (1 + exp(-(v + 35) / 10))


def tau_w(v):
    return 400.0 / (3.3 * exp((v + 35) / 20) + exp(-(v + 35) / 20))

## RTM Bistability with $I_M$: Resting vs. Firing

In [ ]:
def simulate_rtm_with_i_m_bistable(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                    v_k=-100.0, v_na=50.0, v_l=-67.0,
                                    g_m=0.2, i_ext=0.506, t_final=500.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star, w_star = rtm_i_m_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, g_m, i_ext)

    def simulate(v0, m0, h0, n0, w0):
        """Heun/RK2 integration (plain floats) of the RTM model with an added
        M-current, m quasi-static"""
        v, m, h, n, w = v0, m0, h0, n0, w0
        v_trace = np.empty(m_steps + 1)
        v_trace[0] = v
        for k in range(m_steps):
            v_inc = (g_na * m ** 3 * h * (v_na - v) + g_k * n ** 4 * (v_k - v) + g_l * (v_l - v)
                     + g_m * w * (v_k - v) + i_ext) / c
            h_inc = alpha_h(v) * (1 - h) - beta_h(v) * h
            n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n
            w_inc = (w_inf(v) - w) / tau_w(v)

            v_tmp = v + dt05 * v_inc
            m_tmp = m_inf(v_tmp)
            h_tmp = h + dt05 * h_inc
            n_tmp = n + dt05 * n_inc
            w_tmp = w + dt05 * w_inc

            v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                     + g_l * (v_l - v_tmp) + g_m * w_tmp * (v_k - v_tmp) + i_ext) / c
            h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
            n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
            # matches the book's second-stage w_inc, which divides by w(k)
            # (not w_tmp) -- kept as-is for a faithful port
            w_inc = (w_inf(v_tmp) - w) / tau_w(v_tmp)

            v = v + dt * v_inc
            m = m_inf(v)
            h = h + dt * h_inc
            n = n + dt * n_inc
            w = w + dt * w_inc
            v_trace[k + 1] = v
        return v_trace

    v_rest = simulate(v_star, m_star, h_star, n_star, w_star)
    v_fire = simulate(v_star + 1, m_star, h_star, n_star, w_star)
    return v_rest, v_fire, v_star, t_final, dt

In [ ]:
v_rest, v_fire, v_star, t_final, dt = simulate_rtm_with_i_m_bistable()
plot_bistable_rest_fire(v_rest, v_fire, t_final, dt, vlim=(-100, 50))

## RTM Bistability with $I_M$: Gating Variables

In [ ]:
def simulate_rtm_with_i_m_bistable_gates(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                          v_k=-100.0, v_na=50.0, v_l=-67.0,
                                          g_m=0.2, i_ext=0.506, t_final=500.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star, w_star = rtm_i_m_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, g_m, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    w = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0], w[0] = v_star + 1, m_star, h_star, n_star, w_star

    for k in range(m_steps):
        v_inc = (g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_k * n[k] ** 4 * (v_k - v[k]) + g_l * (v_l - v[k])
                 + g_m * w[k] * (v_k - v[k]) + i_ext) / c
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        w_inc = (w_inf(v[k]) - w[k]) / tau_w(v[k])

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        w_tmp = w[k] + dt05 * w_inc

        v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + g_m * w_tmp * (v_k - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        w_inc = (w_inf(v_tmp) - w[k]) / tau_w(v_tmp)

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        w[k + 1] = w[k] + dt * w_inc

    return v, h, n, w, h_star, n_star, w_star, t_final


def plot_rtm_with_i_m_bistable_gates(v, h, n, w, h_star, n_star, w_star, t_final):
    t = np.arange(len(v)) * (t_final / (len(v) - 1))
    fig, ax = plt.subplots(3, figsize=(7, 8), sharex=True)

    ax[0].plot(t, h, '-k', linewidth=2)
    ax[0].plot(t, h_inf(v), '--b', linewidth=2)
    ax[0].plot([0, t_final], [h_star, h_star], '-r', linewidth=2)
    ax[0].set_ylabel('$h$')
    ax[0].set_ylim(0.95, 1)

    ax[1].plot(t, n, '-k', linewidth=2)
    ax[1].plot(t, n_inf(v), '--b', linewidth=2)
    ax[1].plot([0, t_final], [n_star, n_star], '-r', linewidth=2)
    ax[1].set_ylabel('$n$')
    ax[1].set_ylim(0, 0.2)

    ax[2].plot(t, w, '-k', linewidth=2)
    ax[2].plot(t, w_inf(v), '--b', linewidth=2)
    ax[2].plot([0, t_final], [w_star, w_star], '-r', linewidth=2)
    ind = np.where(w < w_star)[0]
    ax[2].plot(t[ind], np.full(len(ind), 0.045), '.m', markersize=8)
    ax[2].set_ylabel('$w$')
    ax[2].set_xlabel('$t$ [ms]')
    ax[2].set_ylim(0.045, 0.06)
    ax[2].set_xlim(0, t_final)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_with_i_m_bistable_gates(*simulate_rtm_with_i_m_bistable_gates())

## RTM Bistability with $I_M$: Limited $w$

Clamps $w$ so it can never fall below $w_\ast$, isolating that this
decline is what lets the trajectory return to repetitive firing.

In [ ]:
def simulate_rtm_with_i_m_limited_w(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                     v_k=-100.0, v_na=50.0, v_l=-67.0,
                                     g_m=0.2, i_ext=0.506, t_final=500.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star, w_star = rtm_i_m_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, g_m, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    w = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0], w[0] = v_star + 1, m_star, h_star, n_star, w_star

    for k in range(m_steps):
        v_inc = (g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_k * n[k] ** 4 * (v_k - v[k]) + g_l * (v_l - v[k])
                 + g_m * w[k] * (v_k - v[k]) + i_ext) / c
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        w_inc = (w_inf(v[k]) - w[k]) / tau_w(v[k])

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        w_tmp = w[k] + dt05 * w_inc

        v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + g_m * w_tmp * (v_k - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp
        w_inc = (w_inf(v_tmp) - w[k]) / tau_w(v_tmp)

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        w[k + 1] = max(w[k] + dt * w_inc, w_star)

    return v, t_final, dt

In [ ]:
plot_voltage_trace_bistable(*simulate_rtm_with_i_m_limited_w(), vlim=(-100, 50))

## Erisir Bistability: Resting vs. Firing

In [ ]:
def erisir_bistable_resting_point(g_k=224.0, g_na=112.0, g_l=0.5, v_k=-90.0, v_na=60.0, v_l=-70.0, i_ext=6.9):
    def alpha_h(v):
        return 0.0035 / exp(v / 24.186)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def beta_h(v):
        return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def f(v):
        return (g_k * n_inf(v) ** 2 * (v_k - v) + g_na * m_inf(v) ** 3 * h_inf(v) * (v_na - v)
                + g_l * (v_l - v) + i_ext) / 1.0

    v_vec = -100 + np.arange(500001) / 500000 * 150
    with np.errstate(invalid='ignore'):
        f_vec = f(v_vec)
    ind = np.where(f_vec[:-1] * f_vec[1:] < 0)[0].min()
    v_star = (v_vec[ind] + v_vec[ind + 1]) / 2
    return v_star, m_inf(v_star), h_inf(v_star), n_inf(v_star)


def simulate_erisir_bistable(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5, v_k=-90.0, v_na=60.0, v_l=-70.0,
                              i_ext=6.9, t_final=40.0, dt=0.01):
    def alpha_h(v):
        return 0.0035 / exp(v / 24.186)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def beta_h(v):
        return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star = erisir_bistable_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, i_ext)

    def simulate(v0, m0, h0, n0):
        """Heun/RK2 integration (plain floats) of the Erisir model, m
        quasi-static, h and n dynamic"""
        v, m, h, n = v0, m0, h0, n0
        v_trace = np.empty(m_steps + 1)
        v_trace[0] = v
        for k in range(m_steps):
            v_inc = (g_k * n ** 2 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
            n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n
            h_inc = alpha_h(v) * (1 - h) - beta_h(v) * h

            v_tmp = v + dt05 * v_inc
            m_tmp = m_inf(v_tmp)
            h_tmp = h + dt05 * h_inc
            n_tmp = n + dt05 * n_inc

            v_inc = (g_k * n_tmp ** 2 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                     + g_l * (v_l - v_tmp) + i_ext) / c
            h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
            n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

            v = v + dt * v_inc
            m = m_inf(v)
            h = h + dt * h_inc
            n = n + dt * n_inc
            v_trace[k + 1] = v
        return v_trace

    v_rest = simulate(v_star, m_star, h_star, n_star)
    v_fire = simulate(v_star + 3, m_star, h_star, n_star)
    return v_rest, v_fire, v_star, t_final, dt

In [ ]:
v_rest, v_fire, v_star, t_final, dt = simulate_erisir_bistable()
plot_bistable_rest_fire(v_rest, v_fire, t_final, dt, vlim=(-95, 55))

## Erisir Bistability: Gating Variables

In [ ]:
def simulate_erisir_bistable_gates(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5, v_k=-90.0, v_na=60.0, v_l=-70.0,
                                    i_ext=6.9, t_final=40.0, dt=0.01):
    def alpha_h(v):
        return 0.0035 / exp(v / 24.186)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def beta_h(v):
        return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star = erisir_bistable_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0] = v_star + 3, m_star, h_star, n_star

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 2 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 2 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc

    return v, m, h, n, m_star, h_star, n_star, t_final


def plot_erisir_bistable_gates(v, m, h, n, m_star, h_star, n_star, t_final):
    def m_inf_e(v):
        alpha_m = 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)
        beta_m = 1.2262 / exp(v / 42.248)
        return alpha_m / (alpha_m + beta_m)

    def h_inf_e(v):
        alpha_h = 0.0035 / exp(v / 24.186)
        beta_h = -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)
        return alpha_h / (alpha_h + beta_h)

    def n_inf_e(v):
        alpha_n = (95 - v) / (exp((95 - v) / 11.8) - 1)
        beta_n = 0.025 / exp(v / 22.222)
        return alpha_n / (alpha_n + beta_n)

    t = np.arange(len(v)) * (t_final / (len(v) - 1))
    fig, ax = plt.subplots(3, figsize=(7, 8), sharex=True)

    ax[0].plot(t, m_inf_e(v), '--b', linewidth=2)
    ax[0].plot([0, t_final], [m_star, m_star], '-r', linewidth=2)
    ax[0].set_ylabel('$m$')
    ax[0].set_ylim(0, 1)

    ax[1].plot(t, h, '-k', linewidth=2)
    ax[1].plot(t, h_inf_e(v), '--b', linewidth=2)
    ax[1].plot([0, t_final], [h_star, h_star], '-r', linewidth=2)
    ax[1].set_ylabel('$h$')
    ax[1].set_ylim(0, 1)

    ax[2].plot(t, n, '-k', linewidth=2)
    ax[2].plot(t, n_inf_e(v), '--b', linewidth=2)
    ax[2].plot([0, t_final], [n_star, n_star], '-r', linewidth=2)
    ax[2].set_ylabel('$n$')
    ax[2].set_xlabel('$t$ [ms]')
    ax[2].set_ylim(0, 1)
    ax[2].set_xlim(0, t_final)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_erisir_bistable_gates(*simulate_erisir_bistable_gates())

## Erisir Bistability: Limited $h$

Clamps $h$ so it can never rise above $h_\ast$, isolating that this
recovery is what lets the trajectory return to rest.

In [ ]:
def simulate_erisir_bistable_limited_h(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5, v_k=-90.0, v_na=60.0, v_l=-70.0,
                                        i_ext=6.9, t_final=40.0, dt=0.01):
    def alpha_h(v):
        return 0.0035 / exp(v / 24.186)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def beta_h(v):
        return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v_star, m_star, h_star, n_star = erisir_bistable_resting_point(g_k, g_na, g_l, v_k, v_na, v_l, i_ext)

    v = np.empty(m_steps + 1)
    m = np.empty(m_steps + 1)
    h = np.empty(m_steps + 1)
    n = np.empty(m_steps + 1)
    v[0], m[0], h[0], n[0] = v_star + 3, m_star, h_star, n_star

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 2 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = alpha_n(v[k]) * (1 - n[k]) - beta_n(v[k]) * n[k]
        h_inc = alpha_h(v[k]) * (1 - h[k]) - beta_h(v[k]) * h[k]

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 2 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = m_inf(v[k + 1])
        h[k + 1] = min(h_star, h[k] + dt * h_inc)
        n[k + 1] = n[k] + dt * n_inc

    return v, t_final, dt

In [ ]:
plot_voltage_trace_bistable(*simulate_erisir_bistable_limited_h(), vlim=(-95, 55))